# 🔷 SUJET 1 — Réduction du Churn Télécom

## Problématique
Une entreprise télécom observe une augmentation des résiliations. Ce notebook propose une **démarche analytique complète** pour :
- 🔍 **Comprendre** les profils clients via la segmentation KMeans
- 🎯 **Anticiper** les départs avec des modèles prédictifs (Logistic Regression, Random Forest, XGBoost)
- 💡 **Agir** avec des recommandations ciblées par segment

---
**Dataset** : `telco_with_clusters.csv` — 7 043 clients, 21 variables (après pipeline KMeans)

## 1. Import des bibliothèques & Chargement des données

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             classification_report, roc_curve)
from xgboost import XGBClassifier

# Style
plt.rcParams['figure.figsize'] = (10, 5)
sns.set_palette("husl")
sns.set_style("whitegrid")

# Chargement des données
df = pd.read_csv('telco_with_clusters.csv')
print(f"Dimensions : {df.shape[0]} clients × {df.shape[1]} variables")
print(f"\nDistribution du Churn :\n{df['Churn'].value_counts()}")
print(f"\nDistribution des Clusters :\n{df['Cluster'].value_counts().sort_index()}")
df.head()

## 2. Analyse Exploratoire des Données (EDA)

> **Objectif** : Comprendre la structure des données, identifier les variables discriminantes du churn, et repérer des patterns clients.

In [ ]:
## 2.1 Distribution du Churn
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Pie chart
churn_counts = df['Churn'].value_counts()
axes[0].pie(churn_counts, labels=['Actif (0)', 'Résilié (1)'],
            autopct='%1.1f%%', colors=['#2ecc71', '#e74c3c'],
            startangle=90, wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[0].set_title('Taux de Churn global', fontsize=14, fontweight='bold')

# Countplot
sns.countplot(data=df, x='Churn', palette=['#2ecc71', '#e74c3c'], ax=axes[1])
axes[1].set_title('Nombre de clients par statut', fontsize=14, fontweight='bold')
axes[1].set_xticklabels(['Actif', 'Résilié'])
axes[1].set_xlabel('Statut client')
axes[1].set_ylabel('Nombre de clients')
for p in axes[1].patches:
    axes[1].annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width()/2., p.get_height()),
                     ha='center', va='bottom', fontsize=12, fontweight='bold')

plt.suptitle('Distribution du Churn', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('churn_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"\n⚠️  Taux de churn : {churn_counts[1]/len(df)*100:.1f}% — classe déséquilibrée à prendre en compte.")

In [ ]:
## 2.2 Variables numériques vs Churn
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

for i, col in enumerate(num_cols):
    # Histogramme
    axes[0, i].hist(df[df['Churn']==0][col], bins=30, alpha=0.7, color='#2ecc71', label='Actif')
    axes[0, i].hist(df[df['Churn']==1][col], bins=30, alpha=0.7, color='#e74c3c', label='Résilié')
    axes[0, i].set_title(f'Distribution de {col}', fontweight='bold')
    axes[0, i].legend()
    axes[0, i].set_xlabel(col)
    axes[0, i].set_ylabel('Fréquence')

    # Boxplot
    df.boxplot(column=col, by='Churn', ax=axes[1, i],
               boxprops=dict(color='steelblue'),
               medianprops=dict(color='red', linewidth=2))
    axes[1, i].set_title(f'Boxplot {col}', fontweight='bold')
    axes[1, i].set_xlabel('Churn (0=Actif, 1=Résilié)')
    axes[1, i].set_ylabel(col)

plt.suptitle('Variables numériques selon le statut Churn', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('numerical_vs_churn.png', dpi=150, bbox_inches='tight')
plt.show()

# Stats clés
print("📊 Statistiques des variables numériques par statut Churn :")
print(df.groupby('Churn')[num_cols].mean().round(2).rename(index={0:'Actif', 1:'Résilié'}))

In [ ]:
## 2.3 Variables catégorielles vs Churn
cat_cols = ['Contract', 'PaymentMethod', 'InternetService', 'MultipleLines', 'TechSupport']

fig, axes = plt.subplots(1, len(cat_cols), figsize=(22, 5))

for i, col in enumerate(cat_cols):
    churn_rate = df.groupby(col)['Churn'].mean().sort_values(ascending=False) * 100
    bars = axes[i].bar(range(len(churn_rate)), churn_rate.values,
                       color=plt.cm.RdYlGn_r(np.linspace(0.2, 0.8, len(churn_rate))))
    axes[i].set_xticks(range(len(churn_rate)))
    axes[i].set_xticklabels(churn_rate.index, rotation=20, ha='right', fontsize=9)
    axes[i].set_title(f'Churn % par\n{col}', fontweight='bold', fontsize=11)
    axes[i].set_ylabel('Taux de churn (%)')
    axes[i].set_ylim(0, 60)
    for bar, val in zip(bars, churn_rate.values):
        axes[i].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.5,
                     f'{val:.1f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.suptitle('Taux de churn par variable catégorielle', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('categorical_vs_churn.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n🔑 Insights clés :")
print(f"  • Contrat Mois/mois vs 2 ans : {df[df['Contract']=='Month-to-month']['Churn'].mean()*100:.1f}% vs {df[df['Contract']=='Two year']['Churn'].mean()*100:.1f}% de churn")
print(f"  • Fibre optique vs DSL : {df[df['InternetService']=='Fiber optic']['Churn'].mean()*100:.1f}% vs {df[df['InternetService']=='DSL']['Churn'].mean()*100:.1f}% de churn")

## 3. Prétraitement des Données

> **Justification** : Encodage des variables catégorielles (Label Encoding pour les binaires, get_dummies pour les multi-classes), normalisation avec StandardScaler pour la régression logistique, split stratifié 80/20 pour conserver le ratio churn.

In [ ]:
## 3.1 Vérification des valeurs manquantes
missing = df.isnull().sum()
print("Valeurs manquantes par colonne :")
print(missing[missing > 0] if missing.sum() > 0 else "  ✅ Aucune valeur manquante détectée.")
print(f"\nValeurs dupliquées : {df.duplicated().sum()}")

## 3.2 Encodage des variables catégorielles
df_model = df.copy()

# Variables catégorielles multi-classes → get_dummies
multi_cat = ['MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup',
             'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies',
             'Contract', 'PaymentMethod']

df_model = pd.get_dummies(df_model, columns=multi_cat, drop_first=True)

print(f"\n✅ Encodage terminé. Dimensions finales : {df_model.shape}")
print(f"   Variables créées : {df_model.shape[1]} features")

## 3.3 Split features / target
X = df_model.drop(columns=['Churn'])
y = df_model['Churn']

## 3.4 Split train / test (80/20 stratifié)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

## 3.5 Normalisation
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# Conserver les DataFrames non-scalés pour Random Forest et XGBoost
print(f"\n📦 Train : {X_train.shape[0]} clients | Test : {X_test.shape[0]} clients")
print(f"   Churn dans le train : {y_train.mean()*100:.1f}%")
print(f"   Churn dans le test  : {y_test.mean()*100:.1f}%")

## 4. Analyse du Churn par Cluster (KMeans)

> **Rappel pipeline** : Le fichier `telco_with_clusters.csv` est le résultat du pipeline KMeans complet (standardisation → détermination k optimal → assignation des clusters). On interprète ici les 3 segments identifiés.

In [ ]:
## 4.1 Profil des clusters — métriques clés
cluster_profile = df.groupby('Cluster').agg(
    Nb_clients=('Churn', 'count'),
    Taux_churn=('Churn', lambda x: f"{x.mean()*100:.1f}%"),
    Tenure_moyen=('tenure', 'mean'),
    Charges_mensuelles_moy=('MonthlyCharges', 'mean'),
    Charges_totales_moy=('TotalCharges', 'mean'),
    Seniors=('SeniorCitizen', lambda x: f"{x.mean()*100:.1f}%")
).round(2)

cluster_profile.index = [f'Cluster {i}' for i in cluster_profile.index]
print("📊 Profil descriptif des clusters KMeans :")
print(cluster_profile.to_string())

## 4.2 Taux de churn par cluster
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Taux de churn par cluster
churn_by_cluster = df.groupby('Cluster')['Churn'].mean() * 100
colors = ['#3498db', '#e67e22', '#9b59b6']
bars = axes[0].bar([f'Cluster {i}' for i in churn_by_cluster.index],
                   churn_by_cluster.values, color=colors, edgecolor='white', linewidth=1.5)
axes[0].axhline(y=df['Churn'].mean()*100, color='red', linestyle='--', label=f'Moy. globale ({df["Churn"].mean()*100:.1f}%)')
axes[0].set_title('Taux de Churn par Cluster', fontweight='bold')
axes[0].set_ylabel('Taux de churn (%)')
axes[0].legend()
for bar, val in zip(bars, churn_by_cluster.values):
    axes[0].text(bar.get_x()+bar.get_width()/2., bar.get_height()+0.5, f'{val:.1f}%',
                 ha='center', fontweight='bold', fontsize=12)

# Tenure moyen par cluster
tenure_by_cluster = df.groupby('Cluster')['tenure'].mean()
axes[1].bar([f'Cluster {i}' for i in tenure_by_cluster.index],
            tenure_by_cluster.values, color=colors, edgecolor='white', linewidth=1.5)
axes[1].set_title('Ancienneté Moyenne par Cluster', fontweight='bold')
axes[1].set_ylabel('Tenure moyen (mois)')
for bar, val in zip(axes[1].patches, tenure_by_cluster.values):
    axes[1].text(bar.get_x()+bar.get_width()/2., bar.get_height()+0.3, f'{val:.1f}m',
                 ha='center', fontweight='bold')

# Charges mensuelles par cluster
charges_by_cluster = df.groupby('Cluster')['MonthlyCharges'].mean()
axes[2].bar([f'Cluster {i}' for i in charges_by_cluster.index],
            charges_by_cluster.values, color=colors, edgecolor='white', linewidth=1.5)
axes[2].set_title('Charges Mensuelles Moy. par Cluster', fontweight='bold')
axes[2].set_ylabel('MonthlyCharges ($)')
for bar, val in zip(axes[2].patches, charges_by_cluster.values):
    axes[2].text(bar.get_x()+bar.get_width()/2., bar.get_height()+0.3, f'${val:.0f}',
                 ha='center', fontweight='bold')

plt.suptitle('Caractéristiques des Clusters KMeans', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('cluster_profiles.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
## 4.3 Répartition des contrats et services par cluster
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Contrats par cluster
contract_cluster = df.groupby(['Cluster', 'Contract']).size().unstack(fill_value=0)
contract_cluster_pct = contract_cluster.div(contract_cluster.sum(axis=1), axis=0) * 100
contract_cluster_pct.index = [f'Cluster {i}' for i in contract_cluster_pct.index]
contract_cluster_pct.plot(kind='bar', ax=axes[0], colormap='Set2', edgecolor='white')
axes[0].set_title('Type de contrat par Cluster', fontweight='bold')
axes[0].set_ylabel('% clients')
axes[0].set_xlabel('')
axes[0].tick_params(axis='x', rotation=0)
axes[0].legend(title='Contrat', bbox_to_anchor=(1, 1))

# Service Internet par cluster
internet_cluster = df.groupby(['Cluster', 'InternetService']).size().unstack(fill_value=0)
internet_cluster_pct = internet_cluster.div(internet_cluster.sum(axis=1), axis=0) * 100
internet_cluster_pct.index = [f'Cluster {i}' for i in internet_cluster_pct.index]
internet_cluster_pct.plot(kind='bar', ax=axes[1], colormap='Set1', edgecolor='white')
axes[1].set_title('Type de service Internet par Cluster', fontweight='bold')
axes[1].set_ylabel('% clients')
axes[1].set_xlabel('')
axes[1].tick_params(axis='x', rotation=0)
axes[1].legend(title='Internet', bbox_to_anchor=(1, 1))

plt.suptitle('Composition des Clusters : Contrats & Services', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('cluster_contracts.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Feature Engineering

> **Justification** : La création de variables dérivées permet d'enrichir la modélisation. Le ratio charges/durée capture la valeur client dans le temps. Le score de services mesure l'engagement produit.

In [ ]:
## 5.1 Création des nouvelles features
df_fe = df.copy()

# Feature 1 : Ratio charge mensuelle / ancienneté
df_fe['charge_per_tenure'] = df_fe['MonthlyCharges'] / (df_fe['tenure'] + 1)

# Feature 2 : Client à haute valeur (top 25% MonthlyCharges)
q75 = df_fe['MonthlyCharges'].quantile(0.75)
df_fe['is_high_value'] = (df_fe['MonthlyCharges'] >= q75).astype(int)

# Feature 3 : Score de services (nombre de services activés)
service_cols_raw = ['PhoneService', 'PaperlessBilling', 'SeniorCitizen']
service_cols_cat = ['MultipleLines', 'OnlineSecurity', 'OnlineBackup',
                    'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']

df_fe['service_count_num'] = df_fe[service_cols_raw].sum(axis=1)
df_fe['service_count_cat'] = df_fe[service_cols_cat].apply(
    lambda col: (col == 'Yes').astype(int)).sum(axis=1)
df_fe['service_count'] = df_fe['service_count_num'] + df_fe['service_count_cat']

print("✅ Nouvelles features créées :")
print(f"   • charge_per_tenure  — moyenne : {df_fe['charge_per_tenure'].mean():.2f}")
print(f"   • is_high_value      — % clients high-value : {df_fe['is_high_value'].mean()*100:.1f}%")
print(f"   • service_count      — moyenne : {df_fe['service_count'].mean():.1f} services/client")

## 5.2 Heatmap de corrélation
num_features = ['tenure', 'MonthlyCharges', 'TotalCharges',
                'charge_per_tenure', 'service_count', 'Cluster', 'Churn']
corr_matrix = df_fe[num_features].corr()

plt.figure(figsize=(9, 7))
mask = np.zeros_like(corr_matrix)
mask[np.triu_indices_from(mask)] = True
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, mask=mask, square=True, linewidths=0.5,
            cbar_kws={"shrink": .8})
plt.title('Matrice de Corrélation (variables numériques)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n🔑 Corrélations avec le Churn :")
print(corr_matrix['Churn'].sort_values(ascending=False).drop('Churn').to_string())

In [ ]:
## 5.3 Reconstruction du dataset modélisation avec les nouvelles features
df_final = df_fe.copy()
df_final = pd.get_dummies(df_final, columns=multi_cat, drop_first=True)

# Supprimer colonnes intermédiaires
df_final.drop(columns=['service_count_num', 'service_count_cat'], inplace=True)

X_full = df_final.drop(columns=['Churn'])
y_full = df_final['Churn']

X_train_f, X_test_f, y_train_f, y_test_f = train_test_split(
    X_full, y_full, test_size=0.2, random_state=42, stratify=y_full)

scaler_f = StandardScaler()
X_train_fs = scaler_f.fit_transform(X_train_f)
X_test_fs  = scaler_f.transform(X_test_f)

print(f"✅ Dataset final : {X_full.shape[1]} features — prêt pour la modélisation")

## 6. Entraînement des Modèles de Prédiction du Churn

> **Justification des choix** :
> - **Régression Logistique** : baseline linéaire, interprétable, adapté aux classes binaires
> - **Random Forest** : robuste aux outliers, pas de normalisation requise, capture les interactions non-linéaires  
> - **XGBoost** : état de l'art sur les données tabulaires, gère le déséquilibre via `scale_pos_weight`
>
> Le **Cluster** est inclus comme feature pour enrichir les modèles avec l'information de segmentation.

In [ ]:
## 6.1 Entraînement — Régression Logistique
print("=" * 55)
print("  MODÈLE 1 : Régression Logistique")
print("=" * 55)

lr = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
lr.fit(X_train_fs, y_train_f)
lr_pred  = lr.predict(X_test_fs)
lr_proba = lr.predict_proba(X_test_fs)[:, 1]

# Cross-validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
lr_cv_auc = cross_val_score(lr, scaler_f.transform(X_full), y_full,
                             cv=cv, scoring='roc_auc').mean()

print(f"\nRapport de classification :\n{classification_report(y_test_f, lr_pred, target_names=['Actif','Résilié'])}")
print(f"ROC-AUC (test)          : {roc_auc_score(y_test_f, lr_proba):.4f}")
print(f"ROC-AUC (CV 5-fold moy) : {lr_cv_auc:.4f}")

In [ ]:
## 6.2 Entraînement — Random Forest
print("=" * 55)
print("  MODÈLE 2 : Random Forest")
print("=" * 55)

rf = RandomForestClassifier(n_estimators=200, random_state=42,
                             class_weight='balanced', n_jobs=-1)
rf.fit(X_train_f, y_train_f)
rf_pred  = rf.predict(X_test_f)
rf_proba = rf.predict_proba(X_test_f)[:, 1]

rf_cv_auc = cross_val_score(rf, X_full, y_full, cv=cv, scoring='roc_auc').mean()

print(f"\nRapport de classification :\n{classification_report(y_test_f, rf_pred, target_names=['Actif','Résilié'])}")
print(f"ROC-AUC (test)          : {roc_auc_score(y_test_f, rf_proba):.4f}")
print(f"ROC-AUC (CV 5-fold moy) : {rf_cv_auc:.4f}")

In [ ]:
## 6.3 Entraînement — XGBoost avec GridSearchCV
print("=" * 55)
print("  MODÈLE 3 : XGBoost (avec optimisation)")
print("=" * 55)

# Ratio pour scale_pos_weight (gestion du déséquilibre)
scale_pw = (y_train_f == 0).sum() / (y_train_f == 1).sum()

param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [3, 5],
    'learning_rate': [0.05, 0.1],
}

xgb_base = XGBClassifier(scale_pos_weight=scale_pw, random_state=42,
                          eval_metric='logloss', verbosity=0)

grid_search = GridSearchCV(xgb_base, param_grid, cv=3, scoring='roc_auc',
                            n_jobs=-1, verbose=0)
grid_search.fit(X_train_f, y_train_f)

best_xgb = grid_search.best_estimator_
xgb_pred  = best_xgb.predict(X_test_f)
xgb_proba = best_xgb.predict_proba(X_test_f)[:, 1]
xgb_cv_auc = cross_val_score(best_xgb, X_full, y_full, cv=cv, scoring='roc_auc').mean()

print(f"Meilleurs hyperparamètres : {grid_search.best_params_}")
print(f"\nRapport de classification :\n{classification_report(y_test_f, xgb_pred, target_names=['Actif','Résilié'])}")
print(f"ROC-AUC (test)          : {roc_auc_score(y_test_f, xgb_proba):.4f}")
print(f"ROC-AUC (CV 5-fold moy) : {xgb_cv_auc:.4f}")

## 7. Évaluation & Comparaison des Modèles

In [ ]:
## 7.1 Tableau comparatif des modèles
models = {
    'Logistic Regression': (lr_pred, lr_proba, lr_cv_auc),
    'Random Forest':       (rf_pred, rf_proba, rf_cv_auc),
    'XGBoost':             (xgb_pred, xgb_proba, xgb_cv_auc),
}

results = []
for name, (pred, proba, cv_auc) in models.items():
    results.append({
        'Modèle': name,
        'Accuracy': round(accuracy_score(y_test_f, pred), 4),
        'Precision': round(precision_score(y_test_f, pred), 4),
        'Recall': round(recall_score(y_test_f, pred), 4),
        'F1-Score': round(f1_score(y_test_f, pred), 4),
        'ROC-AUC (test)': round(roc_auc_score(y_test_f, proba), 4),
        'ROC-AUC (CV)': round(cv_auc, 4),
    })

results_df = pd.DataFrame(results).set_index('Modèle')

# Mise en évidence du meilleur
styled = results_df.style.highlight_max(subset=['Accuracy','Precision','Recall','F1-Score','ROC-AUC (test)','ROC-AUC (CV)'],
                                         color='#c8f7c5', axis=0)
print("📊 Tableau comparatif des modèles :\n")
print(results_df.to_string())
results_df

In [ ]:
## 7.2 Courbes ROC et Matrices de Confusion
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
model_names = list(models.keys())
palette = ['#3498db', '#e67e22', '#9b59b6']

for col, (name, (pred, proba, _)) in enumerate(models.items()):
    # Courbe ROC
    fpr, tpr, _ = roc_curve(y_test_f, proba)
    auc_score   = roc_auc_score(y_test_f, proba)
    axes[0, col].plot(fpr, tpr, color=palette[col], lw=2.5,
                      label=f'AUC = {auc_score:.3f}')
    axes[0, col].plot([0, 1], [0, 1], 'k--', lw=1)
    axes[0, col].fill_between(fpr, tpr, alpha=0.1, color=palette[col])
    axes[0, col].set_title(f'Courbe ROC\n{name}', fontweight='bold')
    axes[0, col].set_xlabel('Taux de Faux Positifs')
    axes[0, col].set_ylabel('Taux de Vrais Positifs')
    axes[0, col].legend(loc='lower right', fontsize=12)
    axes[0, col].set_xlim([0, 1]); axes[0, col].set_ylim([0, 1.02])

    # Matrice de confusion
    cm = confusion_matrix(y_test_f, pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1, col],
                xticklabels=['Actif', 'Résilié'],
                yticklabels=['Actif', 'Résilié'])
    axes[1, col].set_title(f'Matrice de Confusion\n{name}', fontweight='bold')
    axes[1, col].set_xlabel('Prédit')
    axes[1, col].set_ylabel('Réel')

plt.suptitle('Évaluation Comparative des Modèles', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('model_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
## 7.3 Importance des variables (Random Forest & XGBoost)
feature_names = X_full.columns.tolist()

fig, axes = plt.subplots(1, 2, figsize=(18, 8))

for ax, model, title, color in zip(
        axes,
        [rf, best_xgb],
        ['Random Forest', 'XGBoost'],
        ['#e67e22', '#9b59b6']):
    importances = model.feature_importances_
    fi_df = pd.DataFrame({'feature': feature_names, 'importance': importances})
    fi_df = fi_df.sort_values('importance', ascending=True).tail(20)

    ax.barh(fi_df['feature'], fi_df['importance'], color=color, alpha=0.85)
    ax.set_title(f'Top 20 Features — {title}', fontweight='bold', fontsize=13)
    ax.set_xlabel('Importance')

plt.suptitle("Importance des Variables pour la Prédiction du Churn", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

# Top 10 features communes
rf_fi  = pd.Series(rf.feature_importances_, index=feature_names).sort_values(ascending=False).head(10)
xgb_fi = pd.Series(best_xgb.feature_importances_, index=feature_names).sort_values(ascending=False).head(10)

print("🔝 Top 10 features — Random Forest :")
print(rf_fi.to_string())
print("\n🔝 Top 10 features — XGBoost :")
print(xgb_fi.to_string())

## 8. Profilage des Clusters & Interprétation Métier

> Cette section traduit les résultats analytiques en **recommandations actionnables** pour les équipes marketing et fidélisation.

In [ ]:
## 8.1 Tableau de synthèse des clusters
df_fe['service_count_num'] = df_fe[service_cols_raw].sum(axis=1)
df_fe['service_count_cat'] = df_fe[service_cols_cat].apply(
    lambda col: (col == 'Yes').astype(int)).sum(axis=1)
df_fe['service_count'] = df_fe['service_count_num'] + df_fe['service_count_cat']

# Contrat dominant par cluster
contract_dominant = df.groupby('Cluster')['Contract'].agg(
    lambda x: x.value_counts().index[0])

# Tableau complet
cluster_summary = df_fe.groupby('Cluster').agg(
    Nb_clients=('Churn','count'),
    Taux_churn=('Churn', lambda x: round(x.mean()*100, 1)),
    Tenure_moyen=('tenure', lambda x: round(x.mean(), 1)),
    Charges_mensuelles=('MonthlyCharges', lambda x: round(x.mean(), 1)),
    Charges_totales=('TotalCharges', lambda x: round(x.mean(), 1)),
    Nb_services_moy=('service_count', lambda x: round(x.mean(), 1)),
    Pct_seniors=('SeniorCitizen', lambda x: round(x.mean()*100, 1)),
    Pct_high_value=('is_high_value', lambda x: round(x.mean()*100, 1)),
)
cluster_summary['Contrat_dominant'] = contract_dominant

cluster_summary.index = [f'Cluster {i}' for i in cluster_summary.index]
print("📊 Synthèse détaillée des clusters :")
print(cluster_summary.to_string())

In [ ]:
## 8.2 Visualisation radar des clusters
from matplotlib.patches import FancyArrowPatch

# Normalisation des métriques pour le radar
metrics = ['Taux_churn', 'Tenure_moyen', 'Charges_mensuelles', 'Nb_services_moy', 'Pct_seniors']
labels  = ['Taux de\nchurn (%)', 'Ancienneté\n(mois)', 'Charges\nmensuelles ($)', 
           'Nb services', '% Seniors']

radar_data = cluster_summary[metrics].copy()
radar_norm = (radar_data - radar_data.min()) / (radar_data.max() - radar_data.min() + 1e-9)

angles = np.linspace(0, 2 * np.pi, len(metrics), endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(9, 9), subplot_kw=dict(polar=True))
colors = ['#3498db', '#e67e22', '#9b59b6']

for i, (cluster_name, row) in enumerate(radar_norm.iterrows()):
    values = row.tolist() + [row.iloc[0]]
    ax.plot(angles, values, 'o-', linewidth=2.5, color=colors[i], label=cluster_name)
    ax.fill(angles, values, alpha=0.15, color=colors[i])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(labels, fontsize=11)
ax.set_yticks([0.25, 0.5, 0.75, 1.0])
ax.set_yticklabels(['25%', '50%', '75%', '100%'], fontsize=8)
ax.set_title('Profil Radar des Clusters (valeurs normalisées)', fontsize=14, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.15), fontsize=12)

plt.tight_layout()
plt.savefig('cluster_radar.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
## 8.3 Score de risque de churn par client (meilleur modèle)
# Utilisation du meilleur modèle pour scorer tous les clients
X_all_scaled = scaler_f.transform(X_full) if True else X_full

# XGBoost est généralement le meilleur — on utilise ses probabilités
all_proba = best_xgb.predict_proba(X_full)[:, 1]
df_fe['churn_proba'] = all_proba
df_fe['risk_segment'] = pd.cut(all_proba,
                                bins=[0, 0.3, 0.6, 1.0],
                                labels=['🟢 Faible risque', '🟡 Risque modéré', '🔴 Haut risque'])

# Distribution du risque par cluster
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

risk_by_cluster = df_fe.groupby(['Cluster', 'risk_segment']).size().unstack(fill_value=0)
risk_by_cluster_pct = risk_by_cluster.div(risk_by_cluster.sum(axis=1), axis=0) * 100
risk_by_cluster_pct.index = [f'Cluster {i}' for i in risk_by_cluster_pct.index]
risk_by_cluster_pct.plot(kind='bar', stacked=True, ax=axes[0],
                          color=['#2ecc71', '#f39c12', '#e74c3c'], edgecolor='white')
axes[0].set_title('Répartition du Risque de Churn par Cluster', fontweight='bold')
axes[0].set_ylabel('% clients')
axes[0].set_xlabel('')
axes[0].tick_params(axis='x', rotation=0)
axes[0].legend(title='Segment de risque', bbox_to_anchor=(1, 1))

# Distribution des probabilités de churn
for i, c in enumerate(['#3498db', '#e67e22', '#9b59b6']):
    mask = df_fe['Cluster'] == i
    axes[1].hist(df_fe.loc[mask, 'churn_proba'], bins=30, alpha=0.6,
                 color=c, label=f'Cluster {i}', density=True)
axes[1].axvline(x=0.5, color='red', linestyle='--', label='Seuil 0.5')
axes[1].set_title('Distribution des Probabilités de Churn par Cluster', fontweight='bold')
axes[1].set_xlabel('Probabilité de churn prédite')
axes[1].set_ylabel('Densité')
axes[1].legend()

plt.suptitle('Score de Risque XGBoost par Segment', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('churn_risk_by_cluster.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n📊 Nombre de clients par segment de risque :")
print(df_fe['risk_segment'].value_counts().sort_index().to_string())

## 9. Recommandations Métier par Segment

je veux qussi une plqtefor;e### 📊 Résultats réels du pipeline (validés)

| Modèle | Accuracy | F1-Score | ROC-AUC |
|--------|----------|----------|---------|
| Logistic Regression | 0.696 | 0.661 | 0.786 |
| Random Forest | 0.718 | 0.523 | 0.774 |
| **XGBoost** ✅ | **0.698** | **0.662** | **0.787** |

> **XGBoost** est retenu comme meilleur modèle (AUC = 0.787, F1 = 0.662). Ses hyperparamètres optimaux : `learning_rate=0.05, max_depth=3, n_estimators=100`. Le Random Forest a la meilleure accuracy mais un recall faible (classe résiliée sous-détectée) → moins pertinent pour la rétention.

---

### 🔵 Cluster 0 — "Clients à Haut Risque" — 3 060 clients
**Profil réel** : 39.2% de churn — segment le plus volatile.

| Action | Détail | Priorité |
|--------|--------|---------|
| 🎁 Offre de rétention proactive | Remise 10-15% si migration vers contrat 1 an | ⭐⭐⭐ Haute |
| 📞 Appel proactif avant mois 3 | Contacter avant la fin de la période d'essai | ⭐⭐⭐ Haute |
| 🔒 Bundle services | Packager Internet + TechSupport + OnlineSecurity à prix réduit | ⭐⭐ Moyenne |

---

### 🟠 Cluster 1 — "Clients à Risque Modéré" — 1 551 clients
**Profil réel** : 29.2% de churn — segment intermédiaire à surveiller.

| Action | Détail | Priorité |
|--------|--------|---------|
| 🌟 Programme de fidélité | Points/récompenses après 12, 24 mois | ⭐⭐⭐ Haute |
| 📦 Upgrade de services | Proposer Streaming TV/Movies avec essai gratuit 1 mois | ⭐⭐ Moyenne |
| 📧 Communication personnalisée | Newsletters avec usage réel et économies réalisées | ⭐⭐ Moyenne |

---

### 🟣 Cluster 2 — "Clients Fidèles" — 2 432 clients
**Profil réel** : 27.6% de churn — segment le plus stable à valoriser.

| Action | Détail | Priorité |
|--------|--------|---------|
| 👑 Statut VIP / Ambassador | Accès prioritaire au support, avantages exclusifs | ⭐⭐⭐ Haute |
| 🔄 Renouvellement anticipé | Proposer renouvellement avec avantages 3 mois avant échéance | ⭐⭐ Moyenne |
| 💬 NPS & Referral | Exploiter leur satisfaction pour des témoignages et parrainages | ⭐⭐ Moyenne |

---

### 📋 Synthèse : Variables les plus prédictives du Churn

```
┌─────────────────────────────────────────────────────────────────┐
│  TOP FACTEURS DE CHURN (Random Forest + XGBoost concordants)    │
│  1. tenure          — ancienneté faible = fort risque           │
│  2. Contract type   — mois-à-mois = risque x3 vs 2 ans          │
│  3. TotalCharges    — corrélé à l'ancienneté                    │
│  4. MonthlyCharges  — charges élevées + nouveau = danger        │
│  5. TechSupport     — absence de support = facteur de churn     │
│  6. InternetService — Fibre optique = churn le plus élevé       │
│  7. Cluster         — la segmentation enrichit le modèle        │
└─────────────────────────────────────────────────────────────────┘
```

### 🎯 KPIs de suivi recommandés
- **Taux de conversion** contrats M/M → annuels (cible : +5% par trimestre)
- **Taux de churn réel** vs churn prédit (calibration mensuelle du modèle)
- **ROI** des campagnes de rétention par cluster (Cluster 0 prioritaire)
- **Recall sur classe "Résilié"** ≥ 85% (objectif opérationnel)
- **NPS** par segment (benchmark trimestriel)